In [1]:
import pandas as pd, numpy as np
from sqlalchemy import create_engine
engine = create_engine("postgresql+psycopg2://postgres@127.0.0.1:5432/trading")

df = pd.read_sql("SELECT date, adj_close, volume FROM prices WHERE ticker='SPY' ORDER BY date",
                 engine, parse_dates=["date"], index_col="date").astype(float)
px, vol = df["adj_close"], df["volume"]

In [2]:
feat = pd.DataFrame(index=px.index)
ret = px.pct_change()

# RSI 14
delta = px.diff()
gain = delta.clip(lower=0).ewm(alpha=1/14, adjust=False).mean()
loss = (-delta.clip(upper=0)).ewm(alpha=1/14, adjust=False).mean()
feat["rsi14"] = 100 - 100 / (1 + gain / loss)

# MACD histogram
macd = px.ewm(span=12, adjust=False).mean() - px.ewm(span=26, adjust=False).mean()
feat["macd_hist"] = macd - macd.ewm(span=9, adjust=False).mean()

feat["ret_5"]      = px.pct_change(5)
feat["ret_60"]     = px.pct_change(60)
feat["vol_20"]     = ret.rolling(20).std()
feat["dist_ema200"] = px / px.ewm(span=200, adjust=False).mean() - 1
feat["vol_ratio"]  = vol / vol.rolling(20).mean()

In [3]:
feat["target"] = (px.shift(-5) / px - 1 > 0).astype(int)
feat.loc[feat.index[-5:], "target"] = np.nan   # last 5 days have no future yet
feat = feat.iloc[200:].dropna()                # drop warm-up rows
print(feat.shape)
print(feat["target"].mean().round(3), "share of up weeks")
feat.describe().round(3)

(5078, 8)
0.602 share of up weeks


,rsi14,macd_hist,ret_5,ret_60,vol_20,dist_ema200,vol_ratio,target
count,5078.000,5078.000,5078.000,5078.000,5078.000,5078.000,5078.000,5078.000
mean,55.716,0.002,0.002,0.028,0.010,0.040,1.010,0.602
std,11.431,0.902,0.024,0.075,0.007,0.069,0.347,0.490
min,16.803,-6.330,-0.198,-0.410,0.002,-0.369,0.181,0.000
25%,47.609,-0.272,-0.008,-0.005,0.006,0.016,0.779,0.000
50%,57.100,0.015,0.004,0.039,0.008,0.056,0.944,1.000
75%,64.326,0.307,0.015,0.072,0.012,0.083,1.155,1.000
max,87.192,5.567,0.194,0.398,0.060,0.162,3.711,1.000


In [4]:
feat["macd_hist"] = feat["macd_hist"] / px.loc[feat.index]

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

X_cols = ["rsi14", "macd_hist", "ret_5", "ret_60", "vol_20", "dist_ema200", "vol_ratio"]
fit = feat[:"2015-12-31"]
val = feat["2016-01-01":"2019-12-31"]

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model.fit(fit[X_cols], fit["target"])

val_pred = model.predict(val[X_cols])
print("baseline (always up):", round(val["target"].mean(), 3))
print("model accuracy:      ", round((val_pred == val["target"]).mean(), 3))
print("share predicted up:  ", round(val_pred.mean(), 3))

baseline (always up): 0.641
model accuracy:       0.635
share predicted up:   0.994


In [6]:
val_prob = model.predict_proba(val[X_cols])[:, 1]
print(pd.Series(val_prob).describe().round(3))
for th in [0.50, 0.55, 0.60]:
    low = val_prob < th
    print(f"prob < {th}: {low.mean():.1%} of days | up rate on those days: {val['target'][low].mean():.3f}")

count    1006.000
mean        0.591
std         0.031
min         0.408
25%         0.574
50%         0.593
75%         0.611
max         0.697
dtype: float64
prob < 0.5: 0.6% of days | up rate on those days: 1.000
prob < 0.55: 8.5% of days | up rate on those days: 0.640
prob < 0.6: 60.8% of days | up rate on those days: 0.649


In [7]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=300, max_depth=4, min_samples_leaf=50, random_state=42)
rf.fit(fit[X_cols], fit["target"])

rf_prob = rf.predict_proba(val[X_cols])[:, 1]
print("accuracy:", round(((rf_prob > 0.5) == val["target"]).mean(), 3))
for th in [0.55, 0.60]:
    low = rf_prob < th
    print(f"prob < {th}: {low.mean():.1%} of days | up rate: {val['target'][low].mean():.3f}")

accuracy: 0.635
prob < 0.55: 12.4% of days | up rate: 0.672
prob < 0.6: 51.1% of days | up rate: 0.650


In [8]:
def backtest(px, fast, slow, cost_rate=0.0005):
    ema_f = px.ewm(span=fast, adjust=False).mean()
    ema_s = px.ewm(span=slow, adjust=False).mean()
    signal = (ema_f > ema_s).astype(int); signal.iloc[:slow] = 0
    position = signal.shift(1).fillna(0)
    r = px.pct_change().fillna(0)
    return position * r - position.diff().abs().fillna(0) * cost_rate

def metrics(r):
    eq = (1 + r).cumprod(); yrs = len(r) / 252
    return pd.Series({"CAGR": eq.iloc[-1] ** (1 / yrs) - 1,
                      "Sharpe": r.mean() / r.std() * np.sqrt(252),
                      "MaxDD": (eq / eq.cummax() - 1).min()})

test = slice("2020-01-01", "2025-12-31")
out = {}
for t in ["SPY", "QQQ", "GLD"]:
    p = pd.read_sql(f"SELECT date, adj_close FROM prices WHERE ticker='{t}' ORDER BY date",
                    engine, parse_dates=["date"], index_col="date")["adj_close"].astype(float)
    out[f"{t} rule"] = metrics(backtest(p, 75, 100)[test])
    out[f"{t} buy&hold"] = metrics(p.pct_change()[test])
pd.DataFrame(out).T.round(3)

,CAGR,Sharpe,MaxDD
SPY rule,0.072,0.521,-0.304
SPY buy&hold,0.150,0.779,-0.337
QQQ rule,0.196,0.998,-0.286
QQQ buy&hold,0.201,0.851,-0.351
GLD rule,0.151,1.016,-0.248
GLD buy&hold,0.186,1.125,-0.220


In [9]:
import json
from sqlalchemy import text
with engine.begin() as conn:
    for name, m in out.items():
        ticker, kind = name.split(" ")
        conn.execute(text("""
            INSERT INTO backtest_runs (strategy_id, params, train_start, train_end,
                                       test_start, test_end, cagr, sharpe, max_dd)
            VALUES (:sid, CAST(:p AS JSONB), '2005-01-01', '2019-12-31',
                    '2020-01-01', '2025-12-31', :cagr, :sh, :dd)
        """), {"sid": "ema_tuned" if kind == "rule" else "buy_hold",
               "p": json.dumps({"ticker": ticker, "fast": 75, "slow": 100} if kind == "rule" else {"ticker": ticker}),
               "cagr": float(m["CAGR"]), "sh": float(m["Sharpe"]), "dd": float(m["MaxDD"])})

pd.read_sql("SELECT run_id, strategy_id, params, cagr, sharpe, max_dd FROM backtest_runs ORDER BY run_id", engine)

,run_id,strategy_id,params,cagr,sharpe,max_dd
0,1,ema_fixed,"{'fast': 50, 'slow': 200}",NaN,0.689000,NaN
1,2,ema_walkfwd,{'refit': 'yearly'},NaN,0.734000,NaN
2,3,ema_tuned,"{'fast': 75, 'slow': 100, 'status': 'locked'}",NaN,0.791000,NaN
3,4,ema_tuned,"{'fast': 75, 'slow': 100, 'ticker': 'SPY'}",0.071762,0.520917,-0.304465
4,5,buy_hold,{'ticker': 'SPY'},0.150343,0.779324,-0.337173
5,6,ema_tuned,"{'fast': 75, 'slow': 100, 'ticker': 'QQQ'}",0.196352,0.997796,-0.285594
6,7,buy_hold,{'ticker': 'QQQ'},0.201144,0.851049,-0.351187
7,8,ema_tuned,"{'fast': 75, 'slow': 100, 'ticker': 'GLD'}",0.150737,1.016211,-0.248161
8,9,buy_hold,{'ticker': 'GLD'},0.185850,1.125417,-0.220022
